# Упражнение: Residual-блок ResNet с проекцией в skip-связи

## Задание

Реализуйте прямой проход BasicBlock ResNet **в режиме eval** на NumPy. Вход `input.csv` — тензор 2 × 5 × 5
(`channel, row, col, value`). Веса в `weights.json` (раскладка PyTorch: свёртка — `(out, in, kH, kW)`,
без смещений):

```
основной путь:  x → conv1 (3×3, stride 2, padding 1, 2→4) → bn1 → ReLU → conv2 (3×3, stride 1, padding 1, 4→4) → bn2  = F(x)
skip-связь:     x → proj (1×1, stride 2, padding 0, 2→4) → bn_proj                                           = S(x)
выход:          y = ReLU(F(x) + S(x))
```

- Свёртка — как `nn.Conv2d` (кросс-корреляция, без переворота ядра), паддинг нулями:
  out[o, i, j] = Σ_c Σ_a Σ_b W[o, c, a, b] · x_pad[c, i·s + a, j·s + b].
- BatchNorm в eval: (x − running_mean) / √(running_var + eps) · gamma + beta, по каждому каналу; eps — в файле.

## Подсказки

- Размеры: conv1 со stride 2 даёт ⌊(5 + 2 − 3)/2⌋ + 1 = 3, и proj должна дать те же 4 × 3 × 3 — иначе сложить
  F(x) и x нельзя. Именно поэтому в skip-связи стоит проекция, а не identity.
- В режиме eval BN использует **сохранённые** running mean/var, а не статистику текущего входа.
- Одна функция `conv2d(x, W, stride, pad)`: `xp = np.pad(x, ((0, 0), (pad, pad), (pad, pad)))`, затем для
  каждой позиции `np.sum(W[o] * xp[:, i*s:i*s+k, j*s:j*s+k])`; BN — с `[:, None, None]` для broadcast.

## Ожидаемый формат выходного файла

CSV с колонками `channel, row, col, residual, shortcut, out` (residual = F(x), shortcut = S(x), out = y) —
36 строк в порядке channel, row, col; значения округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/resnet
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.